# Basic Virutal Zarr Access

This notebook provides a straightforward demonstration of how to use the
IceChunk-based virtual zarr endpoint to acquire daily NLDAS-3 data across
files without needing to make unneccesary metadata requests, and preventing
the user from manually concatenating data from different files.

In [ ]:
import icechunk
import xarray as xr

## Configure source data and specify subset

In [ ]:
## List endpoints where the virtualized data is stored. This is a security
## step to make sure you don't acquire data from an unknown source.
authorized_urls = [
    "s3://nasa-waterinsight/NLDAS3/forcing/daily/",
    ]
time_slice = slice("2015-12-01", "2016-01-31")
lon_slice = slice(-80,-75)
lat_slice = slice(30, 35)

## Open Icechunk Endpoint and Retrieve Metadata

In [ ]:
## Set up credentials for accessing the s3 bucket and open a connection
repo = icechunk.Repository.open(
    icechunk.s3_storage(
        bucket="nasa-waterinsight",
        prefix="virtual-zarr-store/NLDAS-3-icechunk",
        region="us-west-2",
        anonymous=True,
        ),
    authorize_virtual_chunk_access=icechunk.containers_credentials({
        u:icechunk.s3_credentials(anonymous=True)
        for u in authorized_urls
        })
    )
ses = repo.readonly_session("main")

## acquire general repo metadata by opening the session as a virtual zarr file
ds = xr.open_zarr(ses.store, consolidated=False)

## Restrict to subset and download chunk data

In [ ]:
## use data coordinates to select a subset of the data to actually retrieve
sub = ds.sel(
    time=time_slice,
    lon=lon_slice,
    lat=lat_slice,
    )

## download only the user-defined subset of data.
sub_lw = sub["LWdown"].load()
print(f"Acquired array with shape: {sub_lw.shape}")